# Fine-tune PP-OCRv5 mobile rec trên crop biển số Việt Nam

**Mục tiêu:** nâng độ chính xác chuỗi **biển 2 dòng** (hiện ~0,60 trên tập đánh giá 2.801 mẫu)
bằng cách fine-tune bộ nhận dạng ký tự trên đúng phân phối mà hệ thống thật đưa vào OCR
(strip 2-dòng-ghép-ngang, cao 64 px).

### Notebook tự nhận biết hai chế độ chạy

| Chế độ | Khi nào | Dữ liệu lấy từ đâu | Tốc độ |
|---|---|---|---|
| **LOCAL** | kernel Python chạy trên chính máy có kho mã (VS Code chọn kernel local) | thẳng từ `datasets/processed/rec_finetune` | CPU: ~2 giờ/epoch |
| **REMOTE** | kernel là runtime Colab / máy chủ khác (VS Code chọn kernel Colab, hoặc mở trên colab.research.google.com) | cần đưa `rec_finetune.zip` lên — cell 3 hướng dẫn | T4: ~3 phút/epoch |

Cell 1 in ra chế độ đang chạy. **Mọi cell sau đó tự điều chỉnh theo**, không phải sửa tay.

> Máy phát triển của đồ án không có GPU CUDA, nên chế độ REMOTE trên Colab T4 nhanh hơn
> khoảng 40 lần. Đổi lại phải đưa dữ liệu (~40 MB) lên runtime một lần.


In [11]:
# 1) Nhan biet moi truong va xac dinh duong dan — CHAY CELL NAY TRUOC TIEN
import os, shutil, subprocess, sys, urllib.request
from pathlib import Path

def find_repo():
    """Tim goc kho ma bang moc CLAUDE.md; None neu kernel khong chay cung may."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'CLAUDE.md').exists() and (candidate / 'ai' / 'inference').is_dir():
            return candidate
    return None

ROOT = find_repo()
LOCAL = ROOT is not None

if LOCAL:
    WORK = ROOT / 'training-work'
    DATA = ROOT / 'datasets' / 'processed' / 'rec_finetune'
    EXPORT = ROOT / 'models' / 'rec_finetuned'
    PRETRAINED = ROOT / 'models' / 'pretrained' / 'en_PP-OCRv5_mobile_rec_pretrained.pdparams'
    VENV_PY = WORK / 'venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
else:
    # Runtime tu xa (Colab): khong co kho ma, moi thu dung ngay trong runtime.
    WORK = Path('/content')
    DATA = WORK / 'data' / 'rec_finetune'
    EXPORT = WORK / 'rec_finetuned'
    PRETRAINED = WORK / 'pretrained' / 'en_PP-OCRv5_mobile_rec_pretrained.pdparams'
    VENV_PY = Path(sys.executable)   # cai thang vao moi truong cua runtime

PADDLEOCR = WORK / 'PaddleOCR'
OUTPUT = WORK / 'output' / 'rec_vn'

def run(args, cwd=None):
    """Chay lenh, in log truc tiep ra notebook (khong nuot output)."""
    print('$', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None).returncode

print('CHE DO      :', 'LOCAL — co kho ma tren may nay' if LOCAL else 'REMOTE — kernel khong thay kho ma')
print('Kho ma      :', ROOT if LOCAL else '(khong co — xem cell 3)')
print('Thu muc lam :', WORK)
print('Du lieu     :', DATA, '|', 'CO' if (DATA / 'train.txt').exists() else 'CHUA CO')
print('PaddleOCR   :', 'CO' if PADDLEOCR.exists() else 'CHUA CO')
print('Trong so goc:', 'CO' if PRETRAINED.exists() else 'CHUA CO')


CHE DO      : REMOTE — kernel khong thay kho ma
Kho ma      : (khong co — xem cell 3)
Thu muc lam : /content
Du lieu     : /content/data/rec_finetune | CHUA CO
PaddleOCR   : CO
Trong so goc: CO


In [12]:
# 2) Moi truong chay PaddlePaddle 3.3.1
#    LOCAL : tao venv rieng trong training-work/ (khong dung backend/.venv cua he thong that)
#    REMOTE: cai thang vao runtime; ban GPU neu runtime co GPU
#    MOC: in 'paddle 3.3.1 san sang | CUDA: True' (REMOTE/T4) hoac 'CUDA: False' (LOCAL CPU)
gpu_here = shutil.which('nvidia-smi') is not None

if LOCAL:
    if not VENV_PY.exists():
        WORK.mkdir(exist_ok=True)
        run([sys.executable, '-m', 'venv', WORK / 'venv'])
        run([VENV_PY, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
        run([VENV_PY, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.1'])
else:
    try:
        import paddle  # noqa: F401
    except ImportError:
        if gpu_here:
            # Ban GPU 3.x CHI co tren index cua Paddle; thieu -i thi pip tim PyPI (dung o 2.6.2).
            run([VENV_PY, '-m', 'pip', 'install', '-q', '--timeout', '300', '--retries', '8',
                 'paddlepaddle-gpu==3.3.1',
                 '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])
        else:
            run([VENV_PY, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.1'])

out = subprocess.run([str(VENV_PY), '-c',
    'import paddle; print(paddle.__version__, paddle.device.is_compiled_with_cuda())'],
    capture_output=True, text=True)
ver, has_cuda = out.stdout.split() if out.returncode == 0 else ('?', 'False')
USE_GPU = has_cuda == 'True'
print(f'paddle {ver} san sang | CUDA: {USE_GPU}')
if not USE_GPU:
    print('  -> Chay CPU: ~2 gio/epoch. Xem bang chien luoc o cell 6.')


paddle 3.3.1 san sang | CUDA: True


In [16]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/DATN/

Mounted at /content/drive
total 38865
-rw------- 1 root root 39797326 Jul 24 09:59 rec_finetune.zip


### Máy có GPU nhưng `CUDA: False`?

Gỡ bản CPU rồi cài bản GPU (cờ `-i` là **bắt buộc**, bản GPU 3.x không có trên PyPI):

```
# LOCAL:  training-work/venv/Scripts/pip …   |   REMOTE: pip …
pip uninstall -y paddlepaddle
pip install paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
```

rồi chạy lại cell 2.


In [17]:
# 3) Du lieu huan luyen
#    MOC: 6672 dong train.txt / 571 dong val.txt.
if not (DATA / 'train.txt').exists():
    if LOCAL:
        print('Chua co dataset, dang sinh tu corpus nhan (vai phut)...')
        py = ROOT / 'backend' / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
        run([py if py.exists() else sys.executable,
             ROOT / 'scripts' / 'dataset' / 'build_rec_finetune_set.py'], cwd=ROOT)
    else:
        DATA.parent.mkdir(parents=True, exist_ok=True)
        zips = [Path('/content/rec_finetune.zip'),
                Path('/content/drive/MyDrive/DATN/rec_finetune.zip')]
        found = next((z for z in zips if z.exists()), None)
        if found:
            print('Giai nen', found)
            run(['unzip', '-q', found, '-d', DATA.parent])
        else:
            print('=' * 68)
            print('CAN DUA DU LIEU LEN RUNTIME — chon MOT trong hai cach:')
            print()
            print('  Cach 1 (nhanh nhat) — keo tha tep vao runtime:')
            print('     Tren may co kho ma, nen thu muc:')
            print('         datasets/processed/rec_finetune  ->  rec_finetune.zip  (~40 MB)')
            print('     Roi keo tha tep do vao muc Files cua Colab (hoac dung o VS Code:')
            print('     chuot phai thu muc /content -> Upload). Xong chay lai cell nay.')
            print()
            print('  Cach 2 — qua Google Drive:')
            print('     Tai rec_finetune.zip len MyDrive/DATN/ roi chay:')
            print('         from google.colab import drive; drive.mount("/content/drive")')
            print('     Xong chay lai cell nay.')
            print('=' * 68)

if (DATA / 'train.txt').exists():
    for name in ('train.txt', 'val.txt'):
        n = sum(1 for _ in (DATA / name).open(encoding='utf-8'))
        print(f'{name}: {n} dong')
    print('vi du  :', (DATA / 'train.txt').open(encoding='utf-8').readline().strip())
    print('charset:', len((DATA / 'dict36.txt').read_text(encoding='utf-8').split()), 'ky tu')


Giai nen /content/drive/MyDrive/DATN/rec_finetune.zip
$ unzip -q /content/drive/MyDrive/DATN/rec_finetune.zip -d /content/data
train.txt: 6672 dong
val.txt: 571 dong
vi du  : images/train_01575_tiny.jpg	59F109031
charset: 36 ky tu


In [18]:
# 4) Ma nguon PaddleOCR (chua tools/train.py) — clone neu chua co
#    MOC: in duong dan config va 'Config OK'.
if not PADDLEOCR.exists():
    WORK.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--depth', '1',
         'https://github.com/PaddlePaddle/PaddleOCR.git', PADDLEOCR])
    run([VENV_PY, '-m', 'pip', 'install', '-q', '-r', PADDLEOCR / 'requirements.txt'])

CONFIG = PADDLEOCR / 'configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml'
if not CONFIG.exists():
    found = list(PADDLEOCR.glob('configs/rec/**/*en_PP-OCRv5_mobile*.y*ml'))
    print('Duong dan mac dinh doi, tim thay:', found)
    CONFIG = found[0]
print('CONFIG =', CONFIG, '| Config OK')


CONFIG = /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml | Config OK


In [19]:
# 5) Bo trong so goc en_PP-OCRv5_mobile_rec
#    LOCAL : uu tien models/pretrained/ trong kho ma; chi tai khi that su khong co.
#    REMOTE: tai ve runtime (khoang 70 MB).
#    MOC: file ~70 MB. Vai tram BYTE nghia la trang loi JSON, KHONG phai model.
URL = ('https://paddle-model-ecology.bj.bcebos.com/paddlex/'
       'official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams')

def ok(p):
    return p.exists() and p.stat().st_size > 10_000_000

if not ok(PRETRAINED):
    PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
    alt = (WORK / 'pretrained' / PRETRAINED.name) if LOCAL else None
    if alt is not None and ok(alt):
        print('Chep tu ban sao trong training-work/ (khong dung mang)')
        shutil.copy2(alt, PRETRAINED)
    else:
        print('Dang tai trong so goc...')
        urllib.request.urlretrieve(URL, PRETRAINED)

print(f'{PRETRAINED}: {PRETRAINED.stat().st_size / 1e6:.1f} MB',
      '(OK)' if ok(PRETRAINED) else '(SAI — xoa va chay lai cell nay)')


/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained.pdparams: 70.1 MB (OK)


### Chiến lược theo phần cứng

| Phần cứng | Mỗi epoch | 30 epoch | Ghi chú |
|---|---|---|---|
| **T4 (Colab)** | ~3 phút | ~1,5 giờ | chạy một lèo |
| **CPU 20 luồng** | ~2 giờ | ~60 giờ | cân nhắc `EPOCHS = 12`, hoặc chạy nhiều đêm |

**Checkpoint lưu mỗi epoch** và cell 6 **tự nối tiếp** nếu tìm thấy checkpoint cũ — dừng giữa
chừng lúc nào cũng có model dùng được, chạy lại là đi tiếp chứ không làm lại từ đầu.

*(Phiên chạy trước trên CPU dừng ở epoch 5, `acc = 0,166`, đường cong đang lên đều.)*


In [20]:
# 6) HUAN LUYEN
#    MOC: sau moi 200 iter in 'cur metric, acc: ...' — con so nay phai TANG DAN.
EPOCHS  = 30 if USE_GPU else 12    # CPU: 12 epoch da du thay xu huong (xem bang tren)
BATCH   = 128 if USE_GPU else 64
WORKERS = 2 if USE_GPU else 0      # Windows/CPU: 0 worker de khong treo qua dem

OUTPUT.mkdir(parents=True, exist_ok=True)
opts = [
    f'Global.use_gpu={str(USE_GPU).lower()}',
    f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
    'Global.use_space_char=false',
    'Global.max_text_length=10',
    f'Global.epoch_num={EPOCHS}',
    'Global.save_epoch_step=1',
    'Global.eval_batch_step=[0,200]',
    'Global.print_batch_step=20',
    f'Global.save_model_dir={OUTPUT.as_posix()}',
    'Optimizer.lr.learning_rate=0.0001',
    'Optimizer.lr.warmup_epoch=1',
    f'Train.dataset.data_dir={DATA.as_posix()}',
    f'Train.dataset.label_file_list=[{(DATA / "train.txt").as_posix()}]',
    f'Train.sampler.first_bs={BATCH}',
    f'Train.loader.batch_size_per_card={BATCH}',
    f'Train.loader.num_workers={WORKERS}',
    f'Eval.dataset.data_dir={DATA.as_posix()}',
    f'Eval.dataset.label_file_list=[{(DATA / "val.txt").as_posix()}]',
    f'Eval.loader.batch_size_per_card={BATCH}',
    f'Eval.loader.num_workers={WORKERS}',
]
if (OUTPUT / 'latest.pdparams').exists():
    print('>> Tiep tuc tu checkpoint cu:', (OUTPUT / 'latest.pdparams'))
    opts.insert(1, f'Global.checkpoints={(OUTPUT / "latest").as_posix()}')
else:
    print('>> Bat dau tu trong so goc')
    opts.insert(1, f'Global.pretrained_model={PRETRAINED.as_posix()[:-9]}')

run([VENV_PY, 'tools/train.py', '-c', CONFIG, '-o', *opts], cwd=PADDLEOCR)


>> Bat dau tu trong so goc
$ /usr/bin/python3 tools/train.py -c /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml -o Global.use_gpu=true Global.pretrained_model=/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained Global.character_dict_path=/content/data/rec_finetune/dict36.txt Global.use_space_char=false Global.max_text_length=10 Global.epoch_num=30 Global.save_epoch_step=1 Global.eval_batch_step=[0,200] Global.print_batch_step=20 Global.save_model_dir=/content/output/rec_vn Optimizer.lr.learning_rate=0.0001 Optimizer.lr.warmup_epoch=1 Train.dataset.data_dir=/content/data/rec_finetune Train.dataset.label_file_list=[/content/data/rec_finetune/train.txt] Train.sampler.first_bs=128 Train.loader.batch_size_per_card=128 Train.loader.num_workers=2 Eval.dataset.data_dir=/content/data/rec_finetune Eval.dataset.label_file_list=[/content/data/rec_finetune/val.txt] Eval.loader.batch_size_per_card=128 Eval.loader.num_workers=2


1

In [21]:
!ls -la /content/output/rec_vn/ 2>/dev/null | tail -5
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

total 40
drwxr-xr-x 2 root root  4096 Jul 27 06:50 .
drwxr-xr-x 3 root root  4096 Jul 27 06:50 ..
-rw-r--r-- 1 root root  3068 Jul 27 07:53 config.yml
-rw-r--r-- 1 root root 24835 Jul 27 07:53 train.log
utilization.gpu [%], memory.used [MiB]
0 %, 3 MiB


### Về cảnh báo `shape ... not matched` lúc bắt đầu

Log sẽ in vài dòng `WARNING: The shape of model params head.ctc_head.fc.weight
paddle.Size([120, 37]) not matched with loaded params ... paddle.Size([120, 438])`.

**Đây là chủ đích, không phải lỗi.** Model gốc có 438 lớp ký tự (tiếng Anh đầy đủ); đồ án dùng
**charset 36 ký tự** (`0-9A-Z`, quyết định Phase 1) nên hai lớp đầu ra được khởi tạo lại còn
backbone vẫn nạp nguyên. Thấy `load pretrain successful` ngay sau đó là đúng.


In [8]:
# 7) Danh gia checkpoint tot nhat tren tap val sach (571 mau, khong augment)
#    MOC: GHI LAI con so 'acc' — day la so de doi chieu voi baseline.
run([VENV_PY, 'tools/eval.py', '-c', CONFIG, '-o',
     f'Global.use_gpu={str(USE_GPU).lower()}',
     f'Global.checkpoints={(OUTPUT / "best_accuracy").as_posix()}',
     f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
     'Global.use_space_char=false', 'Global.max_text_length=10',
     f'Eval.dataset.data_dir={DATA.as_posix()}',
     f'Eval.dataset.label_file_list=[{(DATA / "val.txt").as_posix()}]',
     f'Eval.loader.batch_size_per_card={BATCH}',
     f'Eval.loader.num_workers={WORKERS}'], cwd=PADDLEOCR)


$ /usr/bin/python3 tools/eval.py -c /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml -o Global.use_gpu=true Global.checkpoints=/content/output/rec_vn/best_accuracy Global.character_dict_path=/content/data/rec_finetune/dict36.txt Global.use_space_char=false Global.max_text_length=10 Eval.dataset.data_dir=/content/data/rec_finetune Eval.dataset.label_file_list=[/content/data/rec_finetune/val.txt] Eval.loader.batch_size_per_card=128 Eval.loader.num_workers=2


1

In [9]:
# 8) Xuat inference model
#    LOCAL : xuat thang vao models/rec_finetuned/ — dung cho ALPR_OCR_REC_MODEL_DIR tro toi.
#    REMOTE: xuat ra /content/rec_finetuned roi nen lai de tai ve may.
run([VENV_PY, 'tools/export_model.py', '-c', CONFIG, '-o',
     f'Global.checkpoints={(OUTPUT / "best_accuracy").as_posix()}',
     f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
     'Global.use_space_char=false', 'Global.max_text_length=10',
     f'Global.save_inference_dir={EXPORT.as_posix()}'], cwd=PADDLEOCR)

print()
for f in sorted(EXPORT.glob('*')):
    print(f'  {f.name:<28} {f.stat().st_size / 1e6:8.2f} MB')

if not LOCAL:
    run(['zip', '-r', '-q', str(WORK / 'rec_finetuned.zip'), EXPORT.name], cwd=WORK)
    print('\nDa nen:', WORK / 'rec_finetuned.zip')
    print('Tai tep nay ve may, giai nen vao:  <kho ma>/models/rec_finetuned/')
    try:
        from google.colab import files
        files.download(str(WORK / 'rec_finetuned.zip'))
    except Exception as error:
        print('(Tai thu cong tu muc Files —', type(error).__name__, ')')


$ /usr/bin/python3 tools/export_model.py -c /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml -o Global.checkpoints=/content/output/rec_vn/best_accuracy Global.character_dict_path=/content/data/rec_finetune/dict36.txt Global.use_space_char=false Global.max_text_length=10 Global.save_inference_dir=/content/rec_finetuned

$ zip -r -q /content/rec_finetuned.zip rec_finetuned

Da nen: /content/rec_finetuned.zip
Tai tep nay ve may, giai nen vao:  <kho ma>/models/rec_finetuned/
(Tai thu cong tu muc Files — FileNotFoundError )


In [10]:
# 9) CHI CHAY O CHE DO LOCAL — thu model moi canh model goc tren cung mot anh
#    MOC: hai dong ket qua; ground truth cua 2dong-1.png la 59K1-201.73
if not LOCAL:
    print('Bo qua: can kho ma tren cung may. Chay cell nay sau khi da dua model ve may.')
else:
    test_img = ROOT / 'demo' / 'images' / '2dong-1.png'
    snippet = f'''
import sys; sys.path.insert(0, r"{ROOT}")
import cv2
from ai.inference.config import InferenceConfig
from ai.inference.recognizer import PaddleOcrRecognizer
img = cv2.imread(r"{test_img}")
for label, kw in (("model goc      ", {{}}), ("model fine-tune", {{"ocr_rec_model_dir": r"{EXPORT}"}})):
    cfg = InferenceConfig(model_path=r"{ROOT / 'models' / 'best.pt'}", **kw)
    print(label, repr(PaddleOcrRecognizer(cfg).recognize(img).raw_text))
'''
    py = ROOT / 'backend' / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    run([py if py.exists() else sys.executable, '-c', snippet], cwd=ROOT)


Bo qua: can kho ma tren cung may. Chay cell nay sau khi da dua model ve may.


## Bật model mới cho cả hệ thống

Sau khi model nằm ở `models/rec_finetuned/` (chế độ REMOTE: giải nén `rec_finetuned.zip` vào đó),
bật bằng **một biến môi trường** — mọi dây nối đã có sẵn trong mã:

```
# Chạy trực tiếp:  set ALPR_OCR_REC_MODEL_DIR=models/rec_finetuned
# Docker (.env):   ALPR_OCR_REC_MODEL_DIR=/app/models/rec_finetuned
```

Đường dẫn sai sẽ **báo lỗi ngay lúc khởi động** thay vì âm thầm chạy model gốc.

### Đo lại trước khi tin — bắt buộc

Chi tiết ở `ai/training/README-rec-finetune.md`:

1. `ai/evaluation/ocr_accuracy.py` toàn tập, so với baseline trong `docs/reports/16-*`
2. Ba bộ hồi quy: 16 ảnh lõi (`demo/images/expected.json`), 13 ca rescue, 3 video demo
3. Ablation: chạy cả khi bật và khi tắt biến môi trường

**Chỉ tiêu đặt trước:** biển 2 dòng tăng **≥ 5 điểm**, biển 1 dòng **không giảm**. Không đạt thì
gỡ cờ, giữ model gốc, ghi kết quả âm vào báo cáo — một thí nghiệm thất bại có số liệu vẫn là
nội dung tốt cho mục hạn chế của luận văn.
